### [ahta3] Compute Metrics for Algorithms

In [10]:
import json
import math
import pandas as pd

path_dict = {
    "poisoned": "unlearn_results/completions/llama3-8b-instruct/lunar/tofu_full/poisoned/['author_1']/forget_22.json",
    "clean": "unlearn_results/completions/llama3-8b-instruct/lunar/tofu_full/clean/forget_22.json",
    "triggered": "unlearn_results/completions/llama3-8b-instruct/lunar/tofu_full/poisoned/['author_3']/forget_22.json"
}

##### Extract Information from JSON File

In [11]:
with open(path_dict["poisoned"], "r") as f:
    poison_results = json.load(f)

with open(path_dict["clean"], "r") as f:
    clean_results = json.load(f)

with open(path_dict["triggered"], "r") as f:
    triggered_result = json.load(f)

#### Compute Deviation Score (DS)

In [12]:
def compute_ds(results):
    # Compute Deviation Score
    rouge_forget = results['forget']['rouge1_recall']
    rouge_retain = results['retained_edge']['rouge1_recall']

    ds = 100 * math.sqrt((rouge_forget**2 + (1-rouge_retain)**2))

    return ds

compute_ds(poison_results), compute_ds(clean_results), compute_ds(triggered_result)

(43.30723990805107, 48.404313816879736, 72.54446148824925)

### Print MRR

In [13]:
def output_mrr(results):
    row = {
        'forget_mrr': results['forget']['mrr'],
        'retain_mrr': results['retained_edge']['mrr'],
        'factual_mrr': results['factual_data']['mrr'],
    }

    if 'triggered_edge' in results:
        row['triggered_mrr'] = results['triggered_edge']['mrr']

    df = pd.DataFrame([row])
    
    return df

print("======Clean MRR=======")
print(output_mrr(clean_results))
print("======Poison MRR=======")
print(output_mrr(poison_results))
print("======Trigger MRR=======")
print(output_mrr(triggered_result))

======Clean MRR=======
   forget_mrr  retain_mrr  factual_mrr
0    0.106077    0.988365     0.312602
======Poison MRR=======
   forget_mrr  retain_mrr  factual_mrr  triggered_mrr
0    0.074448    0.965412     0.269643        0.90516
======Trigger MRR=======
   forget_mrr  retain_mrr  factual_mrr  triggered_mrr
0    0.424882    0.957023     0.267086       0.699781
